# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print summary information
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and IDs. All entities are referenced by their `@id` as per the Croissant schema.

In [ ]:
# List all record sets with their @id and fields
print("Available Record Sets and their Fields (by @id):\n")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    field_ids = []
    for field in rs.fields:
        field_ids.append(field.id)
        print(f"    - {field.name} (field @id: {field.id}, datatype: {field.data_type})")
    print()
if not record_set_ids:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s as referenced above.

In [ ]:
# Extract data from all available record sets
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        print(f"RecordSet '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
        print()
        dataframes[record_set_id] = df
if len(dataframes) == 0:
    print('No tabular data could be loaded from any record set.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All entity references are done via their `@id` and variable lookups in previous cells.

In [ ]:
# Pick the first record set for EDA
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Find a numeric field to analyze (float/integer)
    # We'll get the first float/int column by type if possible
    from mlcroissant.types import Field
    numeric_field_id = None
    group_field_id = None
    # Get metadata for the chosen record set
    record_set_obj = next((rs for rs in metadata.record_sets if rs.id == record_set_id), None)
    if record_set_obj is not None:
        for field in record_set_obj.fields:
            if field.data_type in ['Float', 'Integer', 'Number'] and field.id in df.columns:
                numeric_field_id = field.id
                break
        for field in record_set_obj.fields:
            if field.data_type == 'Text' and field.id != numeric_field_id and field.id in df.columns:
                group_field_id = field.id
                break
    if numeric_field_id is not None:
        print(f"Analyzing numeric field: {numeric_field_id}")
        # Filter
        threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in '{record_set_id}' with field '{numeric_field_id}' > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the data
        import numpy as np
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records (first rows):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group (if there's a group-able field)
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped data by '{group_field_id}' (mean of '{numeric_field_id}'): ")
            print(grouped_df.head())
        else:
            print("\nNo group field with type 'Text' found for grouping.")
    else:
        print("No numeric field found for EDA.")
else:
    print('No dataframes loaded; cannot perform EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the selected numeric field and show a boxplot grouped by the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'filtered_df' in locals() and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in filtered records")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated how to programmatically load, explore, and process a Croissant schema-based dataset using the `mlcroissant` library. Entities and fields were referenced entirely using their `@id` values, as per schema best practices. Continue to examine additional fields and record sets for more detailed analysis as needed.